**Overview**

Create Block Group and Census Tract dataframes with crime model inputs

    Start with full set of Census Block Groups for US including PR/VI with area and lat/lon
    Add latest Population Estimates
    Add American Community Survey data
    Add Census 2020 data
    Add LODES job data
    Add POI data
    Add NeighborhoodScout schools data

Create municipality dataframe with with crime model inputs

    Aggregate from BG/CT data
    Add American Community Survey data for matching munis
    Check all law enforcement jurisdication geos are supported

Import libraries and set year and directory

In [1]:
import pandas as pd
import pyreadstat
import re
import numpy as np
from gcsfs import GCSFileSystem
import tempfile 

#set year of latest UCR data
year=2024

In [3]:
#run in Google Cloud SDK shell: 
# gcloud auth application-default login

fs = GCSFileSystem(project='clgx-gis-app-dev-06e3', token='google_default')

#Path to GCS buckets
gcs_path = "gs://geospatial-projects/location_inc"

In [4]:
def read_sav_from_gcs(gcs_path: str, fs: GCSFileSystem) -> pd.DataFrame:
    with fs.open(gcs_path, "rb") as gcs_file:
        # Write the content to a temporary file
        with tempfile.NamedTemporaryFile(delete=False) as temp_file:
            temp_file.write(gcs_file.read())
            temp_file.flush()
            temp_file.close()

            # Read the .sav file using pyreadstat
            df, meta = pyreadstat.read_sav(temp_file.name)
    
    return df, meta

In [5]:
def write_sav_to_gcs(df: pd.DataFrame, gcs_path: str, fs: GCSFileSystem) -> None:
    try:
        # Create local empty temporary file
        with tempfile.NamedTemporaryFile(delete=False) as temp_file:
            temp_file.close()
            
            # Add data into local sav file
            pyreadstat.write_sav(df, temp_file.name)
            
            # Load local sav file to GCS
            with open(temp_file.name, "rb") as temp_file_read:
                with fs.open(gcs_path, "wb") as gcs_file:
                    gcs_file.write(temp_file_read.read())
    except Exception as e:
        print(f"Failed to write .sav file to GCS: {e}")

Define function for aggregation using weighted average

In [6]:
def wavg(df,group,weight,values):
    #new dataframe so values can be overwritten
    agg=df.copy()
    #empty dataframe indexed by the group column
    add=pd.DataFrame(columns = [group]).set_index(group)
    #multiply values by weight
    for i in values:
        agg[i] = agg[i] * agg[weight]
        #sum weights and values except if all NaN     
        add=add.join(agg[agg[i].notna()].groupby(group)[[i,weight]].sum(),how='outer')
        #divide values by sum of weights
        add[i] = add[i] / add[weight]
        add.drop(weight,axis=1,inplace=True)
    #return the average values    
    return add

"""wavg(df,group,weight,values) values to average must be entered as a list"""

'wavg(df,group,weight,values) values to average must be entered as a list'

Open full set of 2020 Block Groups and append latest Population Estimates

In [7]:
#import 2020 block group definitions
bg = pd.read_csv(f"{gcs_path}/spatial/census/2020/ns_geographies/block_group_points.csv",
                 converters={'geoid20': str,'statfp20':str})

bg.rename({'geoid20':'bg_key','statefp20':'state_key'},axis=1,inplace=True)
bg['ct_key'] = bg['bg_key'].str[:11]
bg['sqmi']= bg['aland20']* .000000386102
bg['lat'] = bg['intptlat20'].astype(dtype=float)
bg['lon'] = bg['intptlon20'].astype(dtype=float)
bg=bg[['bg_key','ct_key','state_key','sqmi','lat','lon']]

#import bg pop estimates for 50 states + DC + PR
#bg_pop=pd.read_spss(fr'{path}\Demographic Dev\PopEst{year}\bg_popest.sav')
bg_pop, meta = read_sav_from_gcs(f"{gcs_path}/demographic/population_estimates/{year}/bg_popest.sav", fs)
popest=f"popest_{year}"
hpopest=f"popest_{year-5}"

#Get Virgin Islands Census data - still using Census 2010 counts 
vi_cols= 'bg10_key	ct10_key	muni_key	year	Population	PCTAGE_0_4	PCTAGE_18_24	PCTAGE_25_34	PCTAGE_35_54\
	PCTAGE_55_64	HOUSEHOLD	PCTAGE_18_29_MALE	PCTAGE_18_29	PCTAGE_30_44	PCTAGE_45_64	\
PCTAGE_65_UP	AVG_HHSIZE	IN_HOUSEHOLD_PCT	INCARC_PCT	DORMS_PCT	BASE_PCT	GROUP_PHH_PCT	\
ALONE_PCT	NONFAM_PCT	MARWKIDS_PCT	SINGLEMOM_PCT	SINGLEDAD_PCT	VACANT_PCT	\
SEASONAL_PCT	OWN_PCT	OWNCLEAR_PCT	PCTAGE_5_17	PCTAGE_0_17	RNT_PCT'.split()

#vi = pd.read_spss(fr'{path}\Demographic Dev\Census 2010\VI\VI_SF.sav',usecols=vi_cols)
vi, meta = read_sav_from_gcs(f"{gcs_path}/demographic/census/2010/vi/vi_sf.sav", fs)

vi.rename({'bg10_key':'bg_key','ct10_key':'ct_key','Population':hpopest},axis=1,inplace=True)
vi = vi[(vi['year']==2013)&(vi['muni_key'].str[:2]=='78')]
vi['muni_key']='7899' + vi['muni_key'].str[2:5]

#lowercase column names
vi_cols_low = []
for i in vi.columns.to_list():
    vi_cols_low = vi_cols_low + [i.lower()]
vi.columns = vi_cols_low

#add vi block groups to population estimates
bg_pop = pd.concat([bg_pop,vi[vi['bg_key'].str.len()==12]])[['bg_key','popest_2020',popest,hpopest,'pop_ratio']]
bg_pop[popest]= bg_pop[popest].fillna(value=bg_pop[hpopest])
bg_pop['popest_2020'] = bg_pop['popest_2020'].fillna(value=bg_pop[hpopest])
bg_pop['pop_ratio'] = bg_pop['pop_ratio'].fillna(value=0)

#join pop estimates to block groups
bg = bg.join(bg_pop.set_index('bg_key'),on='bg_key')

#replace missing values in VI
for i in ['popest_2020',popest,hpopest]:
    bg[i] = bg[i].ffill()

print('Population Estimates joined for', format((bg[[popest,hpopest]].count().min())/ (
    bg['bg_key'].count()),".3%"),'percent of block groups')

Population Estimates joined for 100.000% percent of block groups


Join American Community Survey data for block groups

In [8]:
#list columns needed from ACS for 50 states+DC+PR
acs_cols = ['col_pct','commtime15_pct','det_pct','gini_index','in_household_pct','alone_pct','households', 'avg_rent','rent_per_room',
            'grandkids_pct','lower_homevalue','seasonal_pct','married_pct','marwkids_pct','md_yrbuilt','group_phh_pct','md_grrent',
                'value_per_room','nonfam_pct','avg_hhsize','md_retax','moved1yr_pct','own_pct','ownclear_pct', 'md_homevalue',
                    'singlemom_pct','singledad_pct','unemp_pct','vacant_pct','rnt_pct','population_acs']
age_cols = 'pctage_0_17	pctage_0_4	pctage_18_24	pctage_18_29	pctage_18_29_male	pctage_25_34	\
    pctage_30_44	pctage_35_54	pctage_45_64	pctage_5_17	pctage_55_64	pctage_65_up'.split()

#open block group level acs
bg_acs, meta = read_sav_from_gcs(f"{gcs_path}/demographic/acs/5/{year-1}/bg_acs.sav", fs)

bg_acs = bg_acs[['bg_key'] + acs_cols + age_cols]

#add VI block groups
vibg = vi[vi['bg_key'].str.len()==12].drop(
    ['ct_key','muni_key','year','dorms_pct','incarc_pct','base_pct'],axis=1).rename(
    {'household':'households',hpopest:'population_acs'},axis=1
)
bg_acs=pd.concat([bg_acs,vibg]).set_index('bg_key')

#join acs to block groups
bg = bg.join(bg_acs,on='bg_key')

print('ACS data joined for', format(
    (bg['population_acs'].count())/ (bg['bg_key'].count()),".3%"),'percent of block groups')

ACS data joined for 99.981% percent of block groups


Create Census Tract level data

In [9]:
ct = bg.groupby('ct_key')[['sqmi',popest,hpopest]].sum()
ct  = ct.join(bg.groupby('ct_key')[['lat','lon']].mean())

#open tract level acs
ct_acs, meta = read_sav_from_gcs(f"{gcs_path}/demographic/acs/5/{year-1}/neighborhood_acs.sav", fs)
ct_acs= ct_acs[['ct_key'] + acs_cols + age_cols]

#add VI tracts
vict = vi[(vi['bg_key'].str.len()==0)&(vi['ct_key'].str.len()==11)].drop(
    ['bg_key','muni_key','year','dorms_pct','incarc_pct','base_pct'],axis=1).rename(
    {'household':'households',hpopest:'population_acs'},axis=1
)
ct_acs=pd.concat([ct_acs,vict]).set_index('ct_key')

ct=ct.join(ct_acs).reset_index()

print('ACS data joined for', format(
    (ct['population_acs'].count())/ (ct['ct_key'].count()),".3%"),'percent of census tracts')


ACS data joined for 99.979% percent of census tracts


Add Census 2020 data for Census Tracts

In [10]:
#add Census 2020 data
pl_2020, meta = read_sav_from_gcs(f"{gcs_path}/demographic/census/2020/redistricting/PL_2020_data_block.sav", fs)
pl_2020=pl_2020[['GEOCODE','CBSA','POP100','P0050003','P0050004','P0050008','P0050009']]  
#aggregate to CT
pl_2020['ct_key']= pl_2020['GEOCODE'].str[:11]
ct_2020 = pl_2020.groupby(['ct_key','CBSA'])[['POP100','P0050003','P0050004','P0050008','P0050009']].sum().reset_index()

ct_2020['incarc_pct'] = 100 * (ct_2020['P0050003']+ct_2020['P0050004'])/ct_2020['POP100']
ct_2020['dorms_pct'] = 100 * (ct_2020['P0050008'])/ct_2020['POP100']
ct_2020['base_pct'] = 100 * (ct_2020['P0050009'])/ct_2020['POP100']
ct_2020.loc[ct_2020['POP100']==0,['incarc_pct','base_pct','dorms_pct']] = 0

#add VI CT
ct_2020 = pd.concat([ct_2020,vi[(vi['bg_key'].str.len()==0)&(vi['ct_key'].str.len()==11)]])[[
    'ct_key','incarc_pct','dorms_pct','base_pct','CBSA'
]]

#recode csba to use state if none defined
ct_2020['CBSA'] = ct_2020['CBSA'].where(
    (ct_2020['CBSA']!='99999') & (ct_2020['CBSA'].notna()), ct_2020['ct_key'].str[:2]+ "999")
ct=ct.join(ct_2020.set_index('ct_key'),on='ct_key')




In [11]:
#add Census Division
div, meta = read_sav_from_gcs(f"{gcs_path}/demographic/census/2020/county_ct_msa.sav", fs)
div = div[['STATE','Division']].drop_duplicates()
ct['state_key'] = ct['ct_key'].str[:2]
ct=ct.join(div.set_index('STATE'),on='state_key')
ct['Division'] = ct['Division'].fillna(value=0)

#percentile by CBSA
ct= ct.join(ct.groupby('CBSA')[['rent_per_room','value_per_room']].rank(pct=True),rsuffix='_rpct')

print('Census 2020 data joined for', format(
    (ct['incarc_pct'].count())/ (ct['ct_key'].count()),".3%"),'percent of census tracts')

Census 2020 data joined for 99.996% percent of census tracts


Add Census 2020 data for block groups

In [12]:
bg=bg.join(ct_2020.set_index('ct_key'),on='ct_key')

bg['state_key'] = bg['ct_key'].str[:2]
bg=bg.join(div.set_index('STATE'),on='state_key')

#percentile by CBSA
bg= bg.join(bg.groupby('CBSA')[['value_per_room','rent_per_room']].rank(pct=True),rsuffix='_rpct')

print('Census 2020 data joined for', format(
    (bg['incarc_pct'].count())/ (bg['bg_key'].count()),".3%"),'percent of block groups')

Census 2020 data joined for 99.999% percent of block groups


Add jobs data to block groups and census tracts

In [13]:
#open latest jobs data
jobs, meta = read_sav_from_gcs(f"{gcs_path}/spatial/lodes/wac_{year-2}/block_jobs.sav", fs)
#jobs = pd.read_spss(fr'{path}\Real Estate Dev\LODES\WAC_{year-2}\data\block_jobs.sav',usecols= 
jobs = jobs[['block_key','C000']]

jobs['bg_key'] = jobs['block_key'].str[:12]    

#join to block groups
bg = bg.join(jobs.groupby('bg_key')['C000'].sum(),on='bg_key')

#add jobs by commute time
comm, meta = read_sav_from_gcs(f"{gcs_path}/spatial/lodes/wac_{year-2}/access_to_jobs_{year-2}.sav", fs)
comm = comm[['bg_key','jobs_5min','jobs_45min']]

#join to block groups
bg = bg.join(comm.set_index('bg_key'),on='bg_key')

print('LODES jobs data joined for', format(
    (bg[['C000','jobs_45min']].count().min())/ (bg['bg_key'].count()),".3%"),'percent of block groups')

#join to census tracts
ct = ct.join(
    bg.groupby('ct_key')['C000'].sum(min_count=1),on='ct_key')

ct = ct.join(
    bg.groupby('ct_key')[['jobs_5min','jobs_45min']].mean(),on='ct_key')


print('LODES jobs data joined for', format(
    (ct[['C000','jobs_45min']].count().min())/ (ct['ct_key'].count()),".3%"),'percent of census tracts')


LODES jobs data joined for 98.088% percent of block groups
LODES jobs data joined for 98.231% percent of census tracts


Schools Data

In [14]:
#get latest test ratings for schools and school districts
scores, meta = read_sav_from_gcs(f"{gcs_path}/school/sy_{year-1}_{year}/bg_edu_scores.sav", fs)
scores = scores[['bg_key','ct_key','usrpct_total_sc','education_score']]

#use district average where school test scores missing
scores['usrpct_total_sc'] = scores['usrpct_total_sc'].where(scores['usrpct_total_sc']>0, scores['education_score'])

#join to block group data
bg = bg.join(scores.set_index('bg_key')['usrpct_total_sc'],on='bg_key')

#use metro average where missing
bg['weight']=bg['households'].where(bg['households']>1,1)

bg= bg.join(wavg(bg,'CBSA','weight',['usrpct_total_sc']),on='CBSA',rsuffix='_mean')

bg['usrpct_total_sc'] = bg['usrpct_total_sc'].where(bg['usrpct_total_sc']>0,bg['usrpct_total_sc_mean'])

print('Schools data joined for', format(
    (bg['usrpct_total_sc'].count())/ (bg['ct_key'].count()),".3%"),'percent of block groups')

#Aggregate to Census Tract
ct = ct.join(wavg(bg,'ct_key','weight',['usrpct_total_sc']),on='ct_key')

print('Schools data joined for', format(
    (ct['usrpct_total_sc'].count())/ (ct['ct_key'].count()),".3%"),'percent of census tracts')


Schools data joined for 98.908% percent of block groups
Schools data joined for 98.814% percent of census tracts


Add POI

In [15]:
poi, meta = read_sav_from_gcs(f"{gcs_path}/demographic/proximity/bg_poi_walk_drive_counts.sav", fs)
poi = poi[['bg_key','city_centers_dist']]
    
#crosswalk from 2010 to 2020 bg boundary
cross, meta = read_sav_from_gcs(f"{gcs_path}/demographic/census/2020/bg_bg_crosswalk.sav", fs)
#cross=pd.read_spss(fr'{path}\Demographic Dev\Census 2020\bg_bg_crosswalk.sav')
cross = cross.join(poi.set_index('bg_key'),on='bg10_key')

bg=bg.join(wavg(cross,'bg_key','pop2020_pct',['city_centers_dist']),on='bg_key')

print('POI data joined for', format(
    (bg['city_centers_dist'].count())/ (bg['bg_key'].count()),".3%"),'percent of block groups')

#Aggregate to Census Tract
ct = ct.join(wavg(bg,'ct_key','weight',['city_centers_dist']),on='ct_key')

print('POI data joined for', format(
    (ct['city_centers_dist'].count())/ (ct['ct_key'].count()),".3%"),'percent of census tracts')

POI data joined for 98.908% percent of block groups
POI data joined for 98.814% percent of census tracts


Population within distance radius

In [16]:
pop_rings, meta = read_sav_from_gcs(f"{gcs_path}/demographic/population_estimates/{year}/pop_rings.sav", fs)
pop_rings = pop_rings[['bg_key','pop_est_5mile','pop_est_50mile','pop_ch_50mile','pop_ch_1mile']].set_index('bg_key')

bg = bg.join(pop_rings,on='bg_key')

print('Population radius data joined for', format(
    (bg['pop_est_5mile'].count())/ (bg['bg_key'].count()),".3%"),'percent of block groups')

#Aggregate to Census Tract
ct = ct.join(wavg(bg,'ct_key','weight',['pop_est_5mile','pop_est_50mile','pop_ch_1mile','pop_ch_50mile']),on='ct_key')

print('Population radius data joined for', format(
    (ct['pop_est_5mile'].count())/ (ct['ct_key'].count()),".3%"),'percent of census tracts')

Population radius data joined for 98.748% percent of block groups
Population radius data joined for 98.440% percent of census tracts


Fill missing with zip code averages

In [17]:
ct_zip = pd.read_excel(fr"{gcs_path}/demographic/relate/TRACT_ZIP_062023.xlsx",dtype={
    'TRACT':str,'ZIP':str
})
#pick predominate zip for each tract
ct_zip_last = ct_zip.sort_values(by=['TRACT','TOT_RATIO']).drop_duplicates(subset='TRACT',keep='last').set_index(
    'TRACT')['ZIP']
ct= ct.join(ct_zip_last,on='ct_key')

#full list of columns to average 
mean_cols = age_cols + acs_cols + ['lat','lon','rent_per_room_rpct',
    'value_per_room_rpct','dorms_pct','Division','usrpct_total_sc','city_centers_dist','incarc_pct',
 'jobs_5min','jobs_45min','pop_est_5mile','pop_est_50mile','pop_ch_50mile', 'pop_ch_1mile']
mean_cols.remove('population_acs')

ct['weight']= ct[popest].replace({0:.01})

print('Complete data for', format(
    (ct[mean_cols].count().min())/ (ct['ct_key'].count()),".3%"),'percent of census tracts') 

ct = ct.join(wavg(ct,'ZIP','weight',mean_cols),on='ZIP',rsuffix='_zip')

for i in mean_cols:
    ct.loc[ct[i].isna(),i] = ct[f'{i}_zip']
    ct.drop(f'{i}_zip',axis=1,inplace=True)

print('Complete data for', format(
    (ct[mean_cols].count().min())/ (ct['ct_key'].count()),".3%"),'percent of census tracts with zip averages')   

#use cbsa avg if zip missing
ct = ct.join(wavg(ct,'CBSA','weight',mean_cols),on='CBSA',rsuffix='_cbsa')

for i in mean_cols:
    ct.loc[ct[i].isna(),i] = ct[f'{i}_cbsa']
    ct.drop(f'{i}_cbsa',axis=1,inplace=True)
 

c:\Users\joglick\AppData\Local\miniforge3\envs\default_python_environment\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Complete data for 93.691% percent of census tracts
Complete data for 98.440% percent of census tracts with zip averages


Municipality level data

In [18]:
#open agency level data
muni, meta = read_sav_from_gcs(f"{gcs_path}/crime/{year}/ucr_muni_sum.sav", fs)

#open agency to geo crosswalk
crosswalk, meta = read_sav_from_gcs(f"{gcs_path}/crime/{year}/ucr_crosswalk.sav", fs)

muni_agency = crosswalk[crosswalk['popest_geo']>0].groupby('muni_key')['akey'].first()

In [19]:
#map to blocks
blocks, meta = read_sav_from_gcs(f"{gcs_path}/demographic/population_estimates/{year}/block_place.sav", fs)
#blocks = pd.read_spss(fr'{path}\Demographic Dev\PopEst{year}\block_place.sav')
#determine which place codes map to crime muni
blocks['muni_key']=blocks['STATE']+blocks['PLACE_CDP']
blocks = blocks.join(muni_agency,on='muni_key')

#match to COUNTY subdivision if not already matched
blocks.loc[blocks['akey'].isna(),'muni_key']=blocks['STATE']+blocks['COUSUB']
blocks = blocks.drop('akey',axis=1).join(muni_agency,on='muni_key')

#match to balance of COUNTY if not already matched
blocks.loc[blocks['akey'].isna(),'muni_key']=blocks['STATE']+'99'+blocks['COUNTY_2020']
#recode small areas lacking coverage
blocks['muni_key'] = blocks['muni_key'].replace({'3499013':'3446380'})
blocks = blocks.drop('akey',axis=1).join(muni_agency,on='muni_key')

#test
blocks[(blocks['akey'].isna())].groupby('muni_key')['POP100'].sum()

muni_key
2599001    0.0
2599005    0.0
2599009    0.0
2599019    0.0
2599023    0.0
2599025    0.0
2699163    0.0
3399015    0.0
3499027    0.0
3699087    0.0
4499005    0.0
5599079    0.0
Name: POP100, dtype: float64

Check population of any areas not matched to police geos. All unmatched areas should have zero population

Sum area and jobs by muni

**Change below:** don't join areas. Already on blocks dataframe

In [20]:
#use nearest agency for any unmathced areas
blocks.loc[blocks['akey'].isna(),'muni_key']=np.nan
blocks.loc[:,['akey','muni_key']] = blocks[['akey','muni_key']].ffill()

#add block area 
#areas, meta = read_sav_from_gcs(f"{gcs_path}/demographic/census/2020/redistricting/PL_2020_data_block.sav", fs)
#areas = areas[['GEOCODE','AREALAND']].set_index('GEOCODE')
#calculate block area in square miles
#blocks=blocks.join(areas,on='GEOID20')
blocks['sqmi']= blocks['AREALAND']* .000000386102

#add jobs counts
blocks= blocks.join(jobs.set_index('block_key'),on='GEOID20')

#aggregate to muni
muni_area = blocks.groupby('akey')[['sqmi','C000']].sum()

#join areas to city list
muni = muni.join(muni_area,on='akey')

muni[muni['sqmi'].isna()]


,akey,muni_key,geo_name,msa_name,statefp,year,population_m,population_ucr,ucr_rape_m,ucr_larceny_m,...,rape_pt_m,robbery_pt_m,assault_pt_m,property_pt_m,burglary_pt_m,larceny_pt_m,mvt_pt_m,metro,sqmi,C000
12409,USVIRGINISLANDSSTCROIX_VI_S,7899010,Virgin Islands 010,,78,2022.0,50601.0,NaN,11.0,118.0,...,0.217387,0.671923,2.134345,6.284461,3.122468,2.33197,0.830023,0.0,NaN,NaN
12410,USVIRGINISLANDSSTTHOMAS_VI_S,multi,multi,,78,2022.0,55804.0,NaN,23.0,70.0,...,0.412157,0.465916,2.490861,3.118056,1.361910,1.25439,0.501756,0.0,NaN,NaN


Move on if only Virgin Islands geos on list above

Aggregate all block group data to muni level

In [21]:
#create bg - muni crosswalk
blocks['bg_key'] = blocks['GEOID20'].str[:12]
bg_muni = blocks.groupby(['bg_key','akey'])['POP100'].sum()

#add vi
#set akeys for VI
vi['akey']='USVIRGINISLANDSSTTHOMAS_VI_S'
vi.loc[vi['muni_key']=='7899010','akey'] = 'USVIRGINISLANDSSTCROIX_VI_S'
vi_bg_muni = vi[vi['bg_key']!=''][['bg_key','akey',hpopest]].rename({hpopest:'POP100'},axis=1)
bg_muni = pd.concat([bg_muni.reset_index(),vi_bg_muni])

bg_muni = bg_muni.join(bg.set_index('bg_key'),on='bg_key')

#weighted average
bg_muni['POP100'] = bg_muni['POP100'].replace({0:.01})
muni_agg = wavg(bg_muni,'akey','POP100',mean_cols)

muni = muni.join(muni_agg,on='akey')

Use muni level reporting from the ACS where available

In [22]:
#open muni level acs
muni_acs, meta = read_sav_from_gcs(f"{gcs_path}/demographic/acs/5/{year-1}/muni_acs.sav", fs)
#muni_acs=pd.read_spss(fr'{path}\Demographic Dev\ACS {year-1} 5yr\muni_acs.sav',usecols=(
muni_acs=muni_acs[['muni_key'] + age_cols + acs_cols]
  
#add VI muni
vi_muni=vi[(vi['bg_key'].str.len()==0)&(vi['ct_key'].str.len()==0)].drop(
    ['bg_key','ct_key','year','dorms_pct','incarc_pct','base_pct'],axis=1).rename(
    {'household':'households',hpopest:'population_acs'},axis=1)

muni_acs=pd.concat([muni_acs,vi_muni]).reset_index(drop=True)    

#adjust muni key for counties
muni_acs.loc[muni_acs['muni_key'].str.len()==5,'muni_key'] = muni_acs['muni_key'].str[:2]+'99'+ muni_acs['muni_key'].str[2:]

#join to muni
muni = muni.join(muni_acs.set_index('muni_key'),on='muni_key',rsuffix='_m')

#use acs reporting at the muni level if police coverage population is at least 75% of acs population
muni_cols = acs_cols + age_cols
muni_cols.remove('population_acs')

for i in muni_cols:
    muni.loc[(muni['population_m']>.75*muni['population_acs'])&(muni[f'{i}_m'].notna()),i] = muni[f'{i}_m']
    muni.drop(f'{i}_m',axis=1,inplace=True)


Derive remaining features and check for completeness


In [23]:
#list all features needed for model
features = ['lat','lon','sqmi','midatlantic','encentral','southatlantic','jobs_5min','jobs_45min','job_density','commtime15_pct','md_retax','population_acs', 'popest',
            'unemp_pct','pop_ch_50mile','pop_ch_1mile','pop_est_5mile','pop_est_50mile','lower_homevalue','md_yrbuilt','avg_hhsize','col_pct','det_pct','gini_index',
            'seasonal_pct','in_household_pct','grandkids_pct','married_pct','value_per_room_rpct','dorms_pct', 'incarc_pct', 'avg_rent','md_homevalue',
            'nonfam_pct','moved1yr_pct','singleparent_pct','vacant_pct','usrpct_total_sc','city_centers_dist','md_grrent','rent_per_room_rpct']

#rename primary population column
muni['popest'] = muni['population_m']
ct['popest']=ct[popest]
bg['popest']=bg[popest]

#derive remaining features
def derive(df):
    df['midatlantic']=0
    df['encentral']=0
    df['southatlantic']=0
    df.loc[df['Division']==2,'midatlantic']=1
    df.loc[df['Division']==3,'encentral']=1
    df.loc[df['Division']==5,'southatlantic']=1
    df['job_density'] = df['C000']/df['sqmi']
    df.loc[df['sqmi']==0,'job_density']=0
    df['singleparent_pct'] = df['singledad_pct']+df['singlemom_pct']
    return df

def check(df,values):
    name =[x for x in globals() if globals()[x] is df][0]
    print('Complete feature data for', format((df[values].count().min())/ (
    df.count().max()),".3%"),'percent of',name)

for i in [bg,ct,muni]:
    derive(i)

#use ct value if bg missing
bg =bg.join(ct.set_index('ct_key')[features],on='ct_key',rsuffix='_ct')

for i in features:
    bg.loc[bg[i].isna(),i] = bg[f'{i}_ct']
    bg.drop(f'{i}_ct',axis=1,inplace=True)

check(bg,features)
check(ct,features)
check(muni,features)

Complete feature data for 98.829% percent of bg
Complete feature data for 98.654% percent of ct
Complete feature data for 98.155% percent of muni


In [24]:
#transform features
#log values 0 and above
log10 =['jobs_5min','job_density','pop_est_5mile','lower_homevalue',
            'seasonal_pct','grandkids_pct','dorms_pct','md_homevalue']

def add_log(df,values):
    for i in values:
        df[f'lg10_{i}'] = np.log10(1+df[i])
    return df

#log values with negatives as low as -100
log_negative = ['pop_ch_50mile','pop_ch_1mile']

def add_log_negative (df,values):
    for i in values:
        df[f'lg10_{i}'] = np.log10(101+df[i])
    return df

#square root
sqrt = ['jobs_45min','md_retax','unemp_pct','pop_est_50mile','avg_hhsize','moved1yr_pct',
        'singleparent_pct','vacant_pct','avg_rent']

def add_sqrt(df,values):
    for i in values:
        df[f'sqrt_{i}'] = np.sqrt(1+df[i])
    return df

#standarize (after running log and square root)
z=['commtime15_pct','md_yrbuilt','col_pct','det_pct','gini_index','married_pct','value_per_room_rpct','city_centers_dist',
   'nonfam_pct','usrpct_total_sc','lg10_jobs_5min','lg10_job_density','lg10_pop_est_5mile','lg10_lower_homevalue',
   'lg10_seasonal_pct','in_household_pct','lg10_grandkids_pct','lg10_dorms_pct','lg10_pop_ch_50mile','lg10_pop_ch_1mile',
   'sqrt_jobs_45min','sqrt_md_retax','sqrt_unemp_pct','sqrt_pop_est_50mile','sqrt_avg_hhsize','sqrt_moved1yr_pct',
   'sqrt_singleparent_pct','sqrt_vacant_pct','rent_per_room_rpct','lg10_md_homevalue','sqrt_avg_rent']
   
def standarize(df,values):
   for i in values:
      df[f'z{i}'] = (df[i] - df[i].mean()) / df[i].std()
   return df

#list of features including transformed
zfeatures=['popest','lat','lon','sqmi','population_acs','incarc_pct','pr','vi','midatlantic','encentral','southatlantic','zcommtime15_pct',
           'zmd_yrbuilt','zcol_pct','zdet_pct','zgini_index','zmarried_pct','zmrpct_val_rm', 'zsqrt_avg_rent','zlg10_md_homevalue',
   'znonfam_pct','zusrpct_total_sc','zlg10_jobs_5min','zlg10_job_density','zlg10_pop_est_5mile','zlg10_lower_homevalue','zcity_centers_dist', 
   'zlg10_seasonal_pct','zin_household_pct','zlg10_grandkids_pct','zlg10_dorms_pct','zlg10_pop_ch_50mile','zlg10_pop_ch_1mile',
   'zsqrt_jobs_45min','zsqrt_md_retax','zsqrt_unemp_pct','zsqrt_pop_est_50mile','zsqrt_avg_hhsize','zsqrt_moved1yr_pct',
   'zsqrt_singleparent_pct','zsqrt_vacant_pct','zmrpct_rent_rm']

#run transformations for each dataframe
for j in [bg,ct,muni]:
    add_log(j,log10)
    add_log_negative(j,log_negative)
    add_sqrt(j,sqrt)
    standarize(j,z)
    j.rename({'zvalue_per_room_rpct':'zmrpct_val_rm','zrent_per_room_rpct':'zmrpct_rent_rm'},axis=1,inplace=True)
    #identifier of PR and VI
    df_name =[x for x in globals() if globals()[x] is j][0]
    j[['pr','vi']]=0
    j.loc[j[f'{df_name}_key'].str[:2]=='72','pr'] = 1
    j.loc[j[f'{df_name}_key'].str[:2]=='78','vi'] = 1
    check(j,zfeatures)

Complete feature data for 98.829% percent of bg
Complete feature data for 98.654% percent of ct
Complete feature data for 98.155% percent of muni


Add census tract data from 5 years prior

In [25]:
ct10_old, meta = read_sav_from_gcs(f"{gcs_path}/crime/{year-5}/bg_ct_muni_data.sav", fs)

old_names= []
for i in ct10_old.columns.to_list():
    old_names = old_names + [i.lower()]

ct10_old.columns = old_names

old_names_lower = 'bg_key ct10_key year zcity_centers_dist zcommtime15_pct zmd_yrbuilt zcol_pct \
zdet_pct zgini_index zmarried_pct zmrpct_val_rm lat lon sqmi zmrpct_rent_rm zlg10_md_homevalue zsqrt_avg_rent \
znonfam_pct zusrpct_total_sc zlg10_jobs_5min zlg10_job_density zlg10_pop_est_5mile zlg10_lower_homevalue \
zlg10_seasonal_pct zin_household_pct zlg10_grandkids_pct zlg10_dorms_pct zlg10_pop_ch_50mile zlg10_pop_ch_1mile \
zsqrt_jobs_45min zsqrt_md_retax zsqrt_unemp_pct zsqrt_pop_est_50mile zsqrt_avg_hhsize zsqrt_moved1yr_pct \
zsqrt_singleparent_pct zsqrt_vacant_pct'.split()

ct10_old = ct10_old[(ct10_old['ct10_key']!='')&(ct10_old['bg_key']=='')&(ct10_old['year']==(year-5))][
      old_names_lower]

#crosswalk
ct_ct_crosswalk, meta = read_sav_from_gcs(f"{gcs_path}/demographic/census/2020/ct20_ct10_crosswalk.sav", fs)
#ct_ct_crosswalk = pd.read_spss(fr'{path}\Demographic Dev\Census 2020\ct20_ct10_crosswalk.sav',usecols=[
ct_ct_crosswalk = ct_ct_crosswalk[['ct10_key','ct20_key','pop2020_pct']].rename({'ct20_key':'ct_key'},axis=1)

ct_ct_crosswalk = ct_ct_crosswalk.join(ct10_old.set_index('ct10_key'),on='ct10_key')

#10aggregate to 2020 ct
ct_old = wavg(ct_ct_crosswalk,'ct_key','pop2020_pct',['year']+zfeatures[11:])

#re-standardize values of older data
for i in zfeatures[11:]:
      ct_old[i] = (ct_old[i] - ct_old[i].mean()) / ct_old[i].std()

#add constants from latest data
ct_old = ct_old.join(ct.set_index('ct_key')[zfeatures[:11]],on='ct_key')      

check(ct_old,zfeatures)

Complete feature data for 98.478% percent of ct_old


Project Census Tract data forward 5 years

In [26]:
#combine old with latest data
ct['year'] = year

ct_years = pd.concat([ct,ct_old.reset_index()]).sort_values(by=['ct_key','year'])[['ct_key','year']+zfeatures]

#add rows for projected data
ct_fut = ct[['ct_key','year']+zfeatures].copy()

ct_fut['year'] = year + 5

ct_years['year'] = ct_years['year'].round()
#project forward
ct_fut = ct_fut.join(ct_old,on='ct_key',rsuffix='_old')
for i in zfeatures:
    ct_fut[i] = (2 * ct_fut[i])- ct_fut[f'{i}_old']
    ct_fut.drop(f'{i}_old',axis=1)

#combine with other years
ct_years = pd.concat([ct_years,ct_fut]).sort_values(by=['ct_key','year'])[['ct_key','year']+zfeatures]

#use current year z score if past or future is missing
ct_years = ct_years.join(ct.set_index('ct_key')[zfeatures],on='ct_key',rsuffix='_cur')

for i in zfeatures:
    ct_years[i] = ct_years[i].fillna(value = ct_years[f'{i}_cur'])
    ct_years.drop(f'{i}_cur',axis=1)

ct_years.groupby('year')[zfeatures].count().min(axis=1)

year
2019.0    84414
2024.0    84277
2029.0    84277
dtype: int64

Format crime rates on city data for modelling steps

In [27]:
#define sets of crime rates
crimes = ['violent','murder','rape','robbery','assault','property','burglary','larceny','mvt']
ucr_counts=[]
ucr_rates =[]
rates_pt = []
rates_change=[]
for i in crimes:
    ucr_counts = ucr_counts + [f'ucr_{i}_m']
    ucr_rates = ucr_rates + [f'ucr_{i}_pt_m']
    rates_pt = rates_pt + [f'{i}_pt_m']
    rates_change = rates_change + [f'pctch_{i}_pt']

#remove any ucr reporting from prior years
muni.loc[muni['year']!=year,ucr_rates] = np.nan
muni.loc[muni['year']!=year,ucr_counts] = np.nan

#indicators of county and metro status
muni[['metropolitan','county']]=0
muni.loc[muni['population_ucr'].isna(),'county']=1
muni.loc[muni['msa_name']!='','metropolitan']=1

#set year to current for all
muni['year']=year

In [28]:
#get city crime totals from 5 years ago
muni_old, meta = read_sav_from_gcs(f"{gcs_path}/crime/{year-5}/ucr_muni_sum.sav", fs)
#muni_old = pd.read_spss(fr'{path}\Crime Dev\UCR {year-5}\ucr_muni_sum.sav')

#reformat agency key
muni_old['sfx']=muni_old['AGENCY_KEY'].apply(lambda x: x.split(',')[-1].lstrip())
muni_old['LEA']=muni_old['AGENCY_KEY'].apply(lambda x: ''.join(x.split(',')[0:-1]))

muni_old['sfx']=muni_old['sfx'].apply(lambda x: re.sub(r'\W+', '',re.sub(r'[(]','_',x)))
muni_old['LEA']=muni_old['LEA'].apply(lambda x: re.sub(r'\W+', '',x))

muni_old['akey']=(muni_old['LEA'] + "_" + muni_old['sfx'])

#lowercase columns
muni_old_cols = []
for i in muni_old.columns.to_list():
    muni_old_cols = muni_old_cols + [i.lower()]

muni_old.columns = muni_old_cols
muni_old.rename({'ucr_theft_pt_m':'ucr_larceny_pt_m'},axis=1,inplace=True)  

#join old rates to city
muni = muni.join(muni_old.set_index('akey')[ucr_rates],on='akey',rsuffix='_old')

muni.reset_index(drop=True,inplace=True)

#calculate change over 5 years
for i in crimes:
    muni[f'pctch_{i}_pt'] = (muni[f'{i}_pt_m']-muni[f'ucr_{i}_pt_m_old'])/muni[f'{i}_pt_m']/5
    muni.loc[muni[f'{i}_pt_m']==0, f'pctch_{i}_pt'] =0 -muni[f'ucr_{i}_pt_m_old']/5
    muni.drop(f'ucr_{i}_pt_m_old',axis=1,inplace=True)

Replace missing with related features where possible

In [29]:
muni['zlg10_lower_homevalue'] = muni['zlg10_lower_homevalue'].fillna(value=muni['zlg10_md_homevalue']).fillna(value=muni['zsqrt_avg_rent'])
muni['zmrpct_val_rm'] = muni['zmrpct_val_rm'].fillna(value=muni['zmrpct_rent_rm']).fillna(value=muni['zlg10_lower_homevalue'])
muni['zsqrt_md_retax'] = muni['zsqrt_md_retax'].fillna(value=muni['zsqrt_avg_rent']).fillna(value=muni['zlg10_lower_homevalue'])


In [30]:
#pyreadstat.write_sav(ct_years,fr'{path}\Crime Dev\UCR {year}\ct_years.sav')
write_sav_to_gcs(ct_years,f"{gcs_path}/crime/{year}/ct_years.sav", fs)
#pyreadstat.write_sav(bg,fr'{path}\Crime Dev\UCR {year}\bg_data.sav')
write_sav_to_gcs(bg,f"{gcs_path}/crime/{year}/bg_data.sav", fs)
#pyreadstat.write_sav(muni,fr'{path}\Crime Dev\UCR {year}\muni_data.sav')
write_sav_to_gcs(muni,f"{gcs_path}/crime/{year}/muni_data.sav", fs)

Create a combined Census Tract - City overlaps dataframe

In [31]:
bg_muni['POP100'] = bg_muni['POP100'].replace({.01:0})
ct_muni = bg_muni.groupby(['akey','ct_key'])['POP100'].sum().rename('population').reset_index()

#keep zero population segments only if total CT population is zero
ct_muni = ct_muni.join(ct_muni.groupby('ct_key')['population'].sum(),on='ct_key',rsuffix='_ct')
ct_muni = ct_muni[(ct_muni['population']>0)|(ct_muni['population_ct']==0)]

#add duplicate rows for past and future 
ct_muni['year']=year
ct_muni_past = ct_muni.copy()
ct_muni_past['year']=year-5
ct_muni_fut = ct_muni.copy()
ct_muni_fut['year']=year+5
ct_muni=pd.concat([ct_muni,ct_muni_past,ct_muni_fut]).sort_values(by=['ct_key']).reset_index(drop=True)

#add rows for city as whole
city = ct_muni.groupby(['akey','year'])['population'].sum().reset_index()
#join all muni-specific columns needed:
city = city.join(muni.set_index('akey')[['geo_name','muni_key']+zfeatures+ucr_counts+ucr_rates+rates_pt],
    on='akey').rename({'geo_name':'name'},axis=1)

In [32]:

#add neighborhood names
ct_names, meta = read_sav_from_gcs(f"{gcs_path}/ns4/current/neighborhoods_ns4.sav", fs)
ct_names = ct_names[['ct_key','name']]

ct_muni= ct_muni.join(ct_names.set_index('ct_key'),on='ct_key')

#add all other ct features
ct_muni = ct_muni.join(ct_years.set_index(['ct_key','year'])[zfeatures],on=['ct_key','year'])

#add city-wide rows.
ct_muni = pd.concat([ct_muni,city.rename({'muni_key':'ct_key'},axis=1)]).sort_values(by=['akey','ct_key','year']).reset_index(drop=True)

#join muni columns needed for all geos
ct_muni = ct_muni.join(muni.set_index('akey')[['muni_key','county','metropolitan','population_m']+rates_change],on='akey')

#add indicator of city row for current year
ct_muni['primary_place']=0
ct_muni.loc[(ct_muni['year']==year)& (ct_muni['ct_key'].str.len() < 11),'primary_place']=1

#adjust popest for census tracts to reflect share within muni
ct_muni = ct_muni.join(ct_muni[ct_muni['population_ct'].notna()].groupby(['akey','year'])['population'].sum(),
             on=['akey','year'],rsuffix='_sum')

ct_muni['popest'] = ct_muni['popest'].where(
    ct_muni['population_m']==0,ct_muni['population'] * ct_muni['population_m'] / ct_muni['population_sum'])

ct_muni.drop(['population','population_sum','population_ct'],axis=1,inplace=True)

Use statewide average for rates of change if missing

In [33]:
#state = pd.read_spss(fr'{path}\Crime Dev\UCR {year}\ucr_state.sav')
state, meta = read_sav_from_gcs(f"{gcs_path}/crime/{year}/ucr_state.sav", fs)
#open state totals for 5 years ago 
state_old, meta = read_sav_from_gcs(f"{gcs_path}/crime/{year-5}/ucr_state.sav", fs)

#lowercase column names then join
old_cols = []
for clm in state_old.columns.to_list():
    old_cols = old_cols + [clm.lower()]
state_old.columns = old_cols
state=state.join(state_old.set_index('stabb'),on='stabb',rsuffix='_old')

#compute state rates of change
for i in crimes:
    state[f'pctch_{i}_pt'] = (state[f'ucr_{i}_rate_s']-state[f'ucr_{i}_rate_s_old'])/state[f'ucr_{i}_rate_s']/5

#join
ct_muni['state_key'] = ct_muni['ct_key'].str[:2]
ct_muni = ct_muni.join(state.set_index('state_key')[rates_change],on='state_key',rsuffix='_mean')

#replace missing and use national average if state missing
for i in rates_change:
    ct_muni[i] = ct_muni[i].fillna(value = ct_muni[f'{i}_mean'])
    ct_muni[i] = ct_muni[i].fillna(value = ct_muni[f'{i}'].mean())
    ct_muni.drop(f'{i}_mean',axis=1,inplace=True)
    #constrain to 2x national average change
    ct_muni[i] = ct_muni[i].where(ct_muni[i]< ct_muni[i].mean()+ 2*abs(ct_muni[i].mean()),ct_muni[i].mean()+ 2*abs(ct_muni[i].mean()))
    ct_muni[i] = ct_muni[i].where(ct_muni[i]> ct_muni[i].mean()- 2*abs(ct_muni[i].mean()),ct_muni[i].mean()- 2*abs(ct_muni[i].mean()))
   


In [34]:
#pyreadstat.write_sav(ct_muni,fr'{path}\Crime Dev\UCR {year}\ct_muni_df.sav')
write_sav_to_gcs(ct_muni,f"{gcs_path}/crime/{year}/ct_muni_df.sav", fs)

Continue with crime_ct_models.ipynb
